In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import copy

print("PyTorch version:", torch.__version__)
print("All libraries imported successfully!")

PyTorch version: 2.11.0+cpu
All libraries imported successfully!


In [6]:
DATASET_PATH = r'C:\Users\Lenovo\Desktop\color'
IMG_SIZE     = 224
BATCH_SIZE   = 16  # increased from 32 → faster training
EPOCHS       = 10
LR           = 0.0001  # lower LR → less overfitting
NUM_CLASSES  = 38
SEED         = 42

print(f"Dataset Path : {DATASET_PATH}")
print(f"Image Size   : {IMG_SIZE}")
print(f"Batch Size   : {BATCH_SIZE}")
print(f"Epochs       : {EPOCHS}")
print(f"Classes      : {NUM_CLASSES}")
print(f"Learning Rate: {LR}")

Dataset Path : C:\Users\Lenovo\Desktop\color
Image Size   : 224
Batch Size   : 16
Epochs       : 10
Classes      : 38
Learning Rate: 0.0001


In [7]:
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(DATASET_PATH)

total      = len(full_dataset)
train_size = int(0.70 * total)
val_size   = int(0.15 * total)
test_size  = total - train_size - val_size

train_data, val_data, test_data = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_data.dataset.transform = train_transforms
val_data.dataset.transform   = val_transforms
test_data.dataset.transform  = val_transforms

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Total  : {total}")
print(f"Train  : {train_size}")
print(f"Val    : {val_size}")
print(f"Test   : {test_size}")

Total  : 54305
Train  : 38013
Val    : 8145
Test   : 8147


In [8]:
# Load pretrained MobileNetV2
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

# Freeze all base layers
for param in model.parameters():
    param.requires_grad = False

# Replace final classification layer
model.classifier = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.last_channel, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, NUM_CLASSES)
)

device = torch.device("cpu")
model  = model.to(device)

print("MobileNetV2 loaded with pretrained ImageNet weights!")
print(f"Using device: {device}")
print(f"Base layers : Frozen ✅")
print(f"Output layer: {NUM_CLASSES} classes ✅")

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to C:\Users\Lenovo/.cache\torch\hub\checkpoints\mobilenet_v2-b0353104.pth


100.0%


MobileNetV2 loaded with pretrained ImageNet weights!
Using device: cpu
Base layers : Frozen ✅
Output layer: 38 classes ✅


In [10]:
# Only train the classifier layers
optimizer = optim.Adam(model.classifier.parameters(), lr=LR)

# Loss function
criterion = nn.CrossEntropyLoss()

# LR Scheduler to prevent overfitting
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=2, factor=0.5
)

print("Loss      : CrossEntropyLoss")
print("Optimizer : Adam (classifier only)")
print("LR        :", LR)
print("Scheduler : ReduceLROnPlateau ✅")

Loss      : CrossEntropyLoss
Optimizer : Adam (classifier only)
LR        : 0.0001
Scheduler : ReduceLROnPlateau ✅


In [11]:
class EarlyStopping:
    def __init__(self, patience=3):
        self.patience   = patience
        self.counter    = 0
        self.best_loss  = None
        self.best_model = None
        self.stop       = False

    def __call__(self, val_loss, model):
        if self.best_loss is None or val_loss < self.best_loss:
            self.best_loss  = val_loss
            self.best_model = copy.deepcopy(model.state_dict())
            self.counter    = 0
        else:
            self.counter += 1
            print(f"Early stopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.stop = True

print("EarlyStopping class defined ✅")

EarlyStopping class defined ✅


In [ ]:
early_stopping = EarlyStopping(patience=3)

train_losses, val_losses         = [], []
train_accuracies, val_accuracies = [], []

for epoch in range(EPOCHS):
    model.train()
    train_loss, train_correct = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss    += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()

    model.eval()
    val_loss, val_correct = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs     = model(images)
            loss        = criterion(outputs, labels)
            val_loss   += loss.item()
            val_correct += (outputs.argmax(1) == labels).sum().item()

    train_acc = train_correct / train_size
    val_acc   = val_correct   / val_size
    t_loss    = train_loss    / len(train_loader)
    v_loss    = val_loss      / len(val_loader)

    train_losses.append(t_loss)
    val_losses.append(v_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Train Loss: {t_loss:.4f} Train Acc: {train_acc:.4f} "
          f"Val Loss: {v_loss:.4f} Val Acc: {val_acc:.4f}")

    scheduler.step(v_loss)
    early_stopping(v_loss, model)
    if early_stopping.stop:
        print("Early stopping triggered!")
        break

model.load_state_dict(early_stopping.best_model)
print("\nTraining complete! Best model loaded ✅")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_accuracies, label='Train Accuracy', color='blue')
axes[0].plot(val_accuracies,   label='Val Accuracy',   color='orange')
axes[0].set_title('Training vs Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(train_losses, label='Train Loss', color='blue')
axes[1].plot(val_losses,   label='Val Loss',   color='orange')
axes[1].set_title('Training vs Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig('transfer_learning_curves.png', dpi=150)
plt.show()
print("Saved: transfer_learning_curves.png ✅")

In [ ]:
model.eval()
test_correct = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs       = model(images)
        test_correct += (outputs.argmax(1) == labels).sum().item()

test_accuracy = test_correct / test_size
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

torch.save(model.state_dict(), 'mobilenet_plantvillage.pth')
print("Model saved: mobilenet_plantvillage.pth ✅")